In [4]:
# Cell 1

# 1) 설치(이미 설치돼 있으면 무시)
!pip -q install datasets==2.20.0 scikit-learn==1.5.2 pandas==2.2.2

# 2) 임포트
import os, random, numpy as np, pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 재현성
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 라벨 기준(보고/지표 표기 용)
id2label = {0:"negative", 1:"neutral", 2:"positive"}
label2id = {"negative":0, "neutral":1, "positive":2}

# 3) 데이터 로드 (신뢰 코드 허용)
ds = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
df = pd.DataFrame(ds["train"])

# 4) 컬럼 표준화 & 클린
# 문장 컬럼 보정
if "sentence" not in df.columns:
    cand = [c for c in df.columns if "sent" in c or "text" in c]
    if not cand:
        raise ValueError(f"문장 컬럼을 찾을 수 없습니다. columns={df.columns.tolist()}")
    df["sentence"] = df[cand[0]]

# 문자열 캐스팅 및 공백 제거
df["sentence"] = df["sentence"].astype(str).str.strip()

# 라벨 처리(정수/문자열 모두 허용)
if "label" not in df.columns:
    raise ValueError("라벨 컬럼(label)이 없습니다.")

raw_label = df["label"]

if pd.api.types.is_integer_dtype(raw_label):
    # 정수 라벨(권장 케이스): 그대로 사용 (0:neg, 1:neu, 2:pos)
    df["label_id"] = raw_label.astype(int)
else:
    # 문자열 라벨 대응
    tmp = raw_label.astype(str).str.strip().str.lower()
    # 다양한 변형 흡수
    canon = {
        "negative":"negative", "neg":"negative", "-1":"negative",
        "neutral":"neutral", "neu":"neutral", "0":"neutral",
        "positive":"positive", "pos":"positive", "1":"positive", "+1":"positive"
    }
    tmp = tmp.map(lambda x: canon.get(x, x))
    # 유효 라벨만 유지
    valid = set(["negative","neutral","positive"])
    df = df[tmp.isin(valid)].copy()
    df["label_id"] = tmp.map(label2id).astype(int)

# 결측/빈 문장 제거
df = df.replace({"": np.nan})
df = df.dropna(subset=["sentence","label_id"]).reset_index(drop=True)

# 방어: 라벨 도메인 확인
uni = sorted(df["label_id"].unique().tolist())
assert set(uni) <= {0,1,2} and len(uni)>=2, f"라벨 도메인 이상: {uni}"

# 5) Stratified split: 0.8/0.1/0.1
texts = df["sentence"].tolist()
labels = df["label_id"].values

X_train_text, X_tmp_text, y_train, y_tmp = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_tmp_text, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp
)

# 6) TF-IDF 적합/변환
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    max_features=20000,
    min_df=2
)
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_val   = vectorizer.transform(X_val_text).toarray()
X_test  = vectorizer.transform(X_test_text).toarray()

# 7) 요약 출력
def dist(y):
    vals, cnts = np.unique(y, return_counts=True)
    return {id2label[int(v)]: int(c) for v,c in zip(vals, cnts)}

print("Financial PhraseBank — sentences_allagree 로드/정리 완료")
print(f"총 문장 수(정리 후): {len(df):,}")
print("레이블 매핑:", id2label)
print("\n[전체 분포]", dist(labels))
print("[Train 분포]", dist(y_train))
print("[Val   분포]", dist(y_val))
print("[Test  분포]", dist(y_test))

print("\nTF-IDF vectorizer:", vectorizer)
print(f"입력 특성 수(n_features): {X_train.shape[1]:,}")
print(f"X_train/X_val/X_test shapes: {X_train.shape}, {X_val.shape}, {X_test.shape}")

print("\n샘플 3개:")
for i in range(min(3, len(X_train_text))):
    s = X_train_text[i].replace("\n"," ")
    print(f"- {s[:160]}{'...' if len(s)>160 else ''}")

# 8) GLOBAL_STORE 저장
GLOBAL_STORE = {
    "X_train": X_train, "y_train": y_train,
    "X_val": X_val,     "y_val": y_val,
    "X_test": X_test,   "y_test": y_test,
    "vectorizer": vectorizer,
    "label2id": label2id, "id2label": id2label,
    "SEED": SEED,
    "texts_raw_df": df
}
print("\n메모리에 GLOBAL_STORE 저장 완료. (다음 단계: 베이스라인 MLP 학습)")


Financial PhraseBank — sentences_allagree 로드/정리 완료
총 문장 수(정리 후): 2,264
레이블 매핑: {0: 'negative', 1: 'neutral', 2: 'positive'}

[전체 분포] {'negative': 303, 'neutral': 1391, 'positive': 570}
[Train 분포] {'negative': 242, 'neutral': 1113, 'positive': 456}
[Val   분포] {'negative': 30, 'neutral': 139, 'positive': 57}
[Test  분포] {'negative': 31, 'neutral': 139, 'positive': 57}

TF-IDF vectorizer: TfidfVectorizer(max_features=20000, min_df=2, ngram_range=(1, 2))
입력 특성 수(n_features): 5,929
X_train/X_val/X_test shapes: (1811, 5929), (226, 5929), (227, 5929)

샘플 3개:
- Merrill Lynch analyst Campbell Morgan upgraded his recommendation on PaperlinX from `` neutral '' to `` buy '' in May .
- Eriikka S+Âderstr+Âm has previously held several positions in finance and control at Nokia Networks including acting as the Business Group Controller and having...
- The webcast may be followed online on the company website at www.ruukki.com/investors .

메모리에 GLOBAL_STORE 저장 완료. (다음 단계: 베이스라인 MLP 학습)


In [6]:
# Cell 2


import os, random, numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

# 재현성 고정
SEED = GLOBAL_STORE.get("SEED", 42)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# 데이터 로드
X_train = GLOBAL_STORE["X_train"]; y_train = GLOBAL_STORE["y_train"]
X_val   = GLOBAL_STORE["X_val"];   y_val   = GLOBAL_STORE["y_val"]
X_test  = GLOBAL_STORE["X_test"];  y_test  = GLOBAL_STORE["y_test"]
id2label = GLOBAL_STORE["id2label"]; label_names = [id2label[i] for i in sorted(id2label.keys())]

input_dim = X_train.shape[1]
n_class   = len(label_names)

# 하이퍼파라미터 (베이스라인)
ACTIVATION = "relu"     # 비교 실험 때 'sigmoid'와 비교
DROPOUT    = 0.5        # 0.3 / 0.5 / 0.7 실험 예정
BATCH_SIZE = 64         # Mini-batch. Batch 학습은 이후 비교에서 별도로 시연
LR         = 1e-3
EPOCHS     = 50

# 모델 정의 함수
def build_mlp(input_dim, n_class, activation="relu", dropout=0.5):
    inputs = tf.keras.Input(shape=(input_dim,), name="tfidf")
    x = tf.keras.layers.Dense(512, activation=activation)(inputs)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(256, activation=activation)(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(n_class, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    return model

model = build_mlp(input_dim, n_class, ACTIVATION, DROPOUT)

optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 콜백
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, verbose=0, min_lr=1e-5
)
csv_logger = tf.keras.callbacks.CSVLogger("baseline_train_log.csv", append=False)

# 학습
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr, csv_logger],
    verbose=0
)

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"Baseline 학습 완료 — best_epoch={best_epoch}, "
      f"final_val_loss={history.history['val_loss'][best_epoch-1]:.4f}")

# 평가 함수
def evaluate_split(name, X, y, model):
    prob = model.predict(X, verbose=0)
    y_pred = prob.argmax(axis=1)
    acc = accuracy_score(y, y_pred)
    macro_f1 = f1_score(y, y_pred, average="macro")
    report = classification_report(y, y_pred, target_names=label_names, digits=4, zero_division=0)
    cm = confusion_matrix(y, y_pred, labels=list(range(n_class)))
    return {"name":name, "acc":acc, "macro_f1":macro_f1, "report":report, "cm":cm, "y_pred":y_pred}

val_metrics  = evaluate_split("VAL",  X_val,  y_val,  model)
test_metrics = evaluate_split("TEST", X_test, y_test, model)

# 간단 표 출력
def print_summary(val_m, test_m):
    print("\n=== 성능 요약 (Baseline: Adam, ReLU, Dropout=0.5, mini-batch=64) ===")
    print(f"VAL  — Acc: {val_m['acc']:.4f} | Macro-F1: {val_m['macro_f1']:.4f}")
    print(f"TEST — Acc: {test_m['acc']:.4f} | Macro-F1: {test_m['macro_f1']:.4f}")
    print("\n[VAL Classification Report]\n" + val_m["report"])
    print("[VAL Confusion Matrix]\n", val_m["cm"])
    print("\n[TEST Classification Report]\n" + test_m["report"])
    print("[TEST Confusion Matrix]\n", test_m["cm"])

print_summary(val_metrics, test_metrics)

# GLOBAL_STORE에 저장(후속 비교 실험에서 재사용)
GLOBAL_STORE["baseline"] = {
    "params": {
        "activation": ACTIVATION,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "lr": LR,
        "epochs": EPOCHS,
        "best_epoch": best_epoch
    },
    "val": val_metrics,
    "test": test_metrics,
    "model": model
}

print("\n baseline 결과와 모델을 GLOBAL_STORE['baseline']에 저장했습니다.")


Baseline 학습 완료 — best_epoch=10, final_val_loss=0.1823

=== 성능 요약 (Baseline: Adam, ReLU, Dropout=0.5, mini-batch=64) ===
VAL  — Acc: 0.9513 | Macro-F1: 0.9287
TEST — Acc: 0.8899 | Macro-F1: 0.8444

[VAL Classification Report]
              precision    recall  f1-score   support

    negative     0.8966    0.8667    0.8814        30
     neutral     0.9714    0.9784    0.9749       139
    positive     0.9298    0.9298    0.9298        57

    accuracy                         0.9513       226
   macro avg     0.9326    0.9250    0.9287       226
weighted avg     0.9510    0.9513    0.9511       226

[VAL Confusion Matrix]
 [[ 26   0   4]
 [  3 136   0]
 [  0   4  53]]

[TEST Classification Report]
              precision    recall  f1-score   support

    negative     0.8519    0.7419    0.7931        31
     neutral     0.9247    0.9712    0.9474       139
    positive     0.8148    0.7719    0.7928        57

    accuracy                         0.8899       227
   macro avg     0.863

In [8]:
# Cell 3


import random, numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score

assert "GLOBAL_STORE" in globals(), "먼저 Cell 1C와 Cell 2를 실행해 GLOBAL_STORE를 준비하세요."

# 재현성
SEED = GLOBAL_STORE.get("SEED", 42)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# 데이터
X_train = GLOBAL_STORE["X_train"]; y_train = GLOBAL_STORE["y_train"]
X_val   = GLOBAL_STORE["X_val"];   y_val   = GLOBAL_STORE["y_val"]
X_test  = GLOBAL_STORE["X_test"];  y_test  = GLOBAL_STORE["y_test"]
id2label = GLOBAL_STORE["id2label"]; label_names = [id2label[i] for i in sorted(id2label.keys())]

input_dim = X_train.shape[1]
n_class   = len(label_names)

# 고정 하이퍼
ACTIVATION = "relu"
DROPOUT    = 0.5
BATCH_SIZE = 64
EPOCHS     = 50

def build_mlp(input_dim, n_class, activation="relu", dropout=0.5):
    inputs = tf.keras.Input(shape=(input_dim,), name="tfidf")
    x = tf.keras.layers.Dense(512, activation=activation)(inputs)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(256, activation=activation)(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(n_class, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

def evaluate_split(X, y, model):
    prob = model.predict(X, verbose=0)
    y_pred = prob.argmax(axis=1)
    return accuracy_score(y, y_pred), f1_score(y, y_pred, average="macro")

optz = {
    "SGD": tf.keras.optimizers.SGD(learning_rate=1e-2),
    "Momentum": tf.keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9),
    "RMSProp": tf.keras.optimizers.RMSprop(learning_rate=1e-3),
    "Adam": tf.keras.optimizers.Adam(learning_rate=1e-3),
}

records = []
models_kept = {}

for name, optimizer in optz.items():
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = build_mlp(input_dim, n_class, ACTIVATION, DROPOUT)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, verbose=0, min_lr=1e-5
    )

    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )

    val_acc, val_f1 = evaluate_split(X_val, y_val, model)
    test_acc, test_f1 = evaluate_split(X_test, y_test, model)
    best_epoch = np.argmin(hist.history["val_loss"]) + 1

    records.append({
        "optimizer": name,
        "best_epoch": best_epoch,
        "val_acc": round(val_acc, 4),
        "val_macroF1": round(val_f1, 4),
        "test_acc": round(test_acc, 4),
        "test_macroF1": round(test_f1, 4),
    })
    models_kept[name] = model

df = pd.DataFrame(records).sort_values(by=["val_macroF1","val_acc"], ascending=False).reset_index(drop=True)
best_name = df.loc[0, "optimizer"]

print("=== Optimizer Comparison (ReLU, Dropout=0.5, batch=64) ===")
print(df.to_string(index=False))
print(f"\nBest by Val Macro-F1: {best_name}")

# GLOBAL_STORE에 저장
GLOBAL_STORE["optimizer_cmp"] = {
    "results": df,
    "best": best_name,
    "models": models_kept
}
print("\noptimizer 비교 결과를 GLOBAL_STORE['optimizer_cmp']에 저장했습니다.")


=== Optimizer Comparison (ReLU, Dropout=0.5, batch=64) ===
optimizer  best_epoch  val_acc  val_macroF1  test_acc  test_macroF1
  RMSProp          10   0.9469       0.9274    0.9031        0.8507
     Adam           5   0.9469       0.9224    0.8899        0.8444
 Momentum          49   0.9292       0.8892    0.9075        0.8589
      SGD          50   0.6150       0.2539    0.6123        0.2532

Best by Val Macro-F1: RMSProp

optimizer 비교 결과를 GLOBAL_STORE['optimizer_cmp']에 저장했습니다.


In [9]:
# Cell 4


import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report

assert "GLOBAL_STORE" in globals(), "먼저 이전 셀들을 실행해 GLOBAL_STORE를 준비하세요."

# 재현성
SEED = GLOBAL_STORE.get("SEED", 42)
np.random.seed(SEED); tf.random.set_seed(SEED)

# 데이터
X_train = GLOBAL_STORE["X_train"]; y_train = GLOBAL_STORE["y_train"]
X_val   = GLOBAL_STORE["X_val"];   y_val   = GLOBAL_STORE["y_val"]
X_test  = GLOBAL_STORE["X_test"];  y_test  = GLOBAL_STORE["y_test"]

input_dim = X_train.shape[1]
n_class   = 3

# 고정 하이퍼
DROPOUT    = 0.5
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3

def build_mlp(input_dim, n_class, activation="relu", dropout=0.5):
    inputs = tf.keras.Input(shape=(input_dim,), name="tfidf")
    x = tf.keras.layers.Dense(512, activation=activation)(inputs)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(256, activation=activation)(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(n_class, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

def train_eval(activation):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = build_mlp(input_dim, n_class, activation=activation, dropout=DROPOUT)
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    rlrop = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=0)

    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlrop],
        verbose=0
    )
    best_epoch = int(np.argmin(hist.history["val_loss"]) + 1)

    def eval_split(X, y):
        prob = model.predict(X, verbose=0)
        y_pred = prob.argmax(axis=1)
        return (
            accuracy_score(y, y_pred),
            f1_score(y, y_pred, average="macro"),
            classification_report(y, y_pred, digits=4, zero_division=0)
        )

    v_acc, v_f1, v_rep = eval_split(X_val, y_val)
    t_acc, t_f1, t_rep = eval_split(X_test, y_test)

    return {
        "activation": activation,
        "best_epoch": best_epoch,
        "val_acc": round(v_acc,4), "val_macroF1": round(v_f1,4),
        "test_acc": round(t_acc,4), "test_macroF1": round(t_f1,4),
        "val_report": v_rep, "test_report": t_rep, "model": model
    }

results = []
kept = {}

for act in ["relu", "sigmoid"]:
    out = train_eval(act)
    results.append({k: out[k] for k in ["activation","best_epoch","val_acc","val_macroF1","test_acc","test_macroF1"]})
    kept[act] = out

df = pd.DataFrame(results).sort_values(by=["val_macroF1","val_acc"], ascending=False).reset_index(drop=True)
winner = df.loc[0, "activation"]

print("=== Activation Comparison (optimizer=RMSProp, dropout=0.5, batch=64) ===")
print(df.to_string(index=False))
print(f"\nBest by Val Macro-F1: {winner}")

# 상세 리포트(선택적으로 확인)
print("\n[VAL report — ReLU]\n", kept["relu"]["val_report"])
print("[TEST report — ReLU]\n", kept["relu"]["test_report"])
print("\n[VAL report — Sigmoid]\n", kept["sigmoid"]["val_report"])
print("[TEST report — Sigmoid]\n", kept["sigmoid"]["test_report"])

# 저장
GLOBAL_STORE["activation_cmp"] = {
    "results": df,
    "best": winner,
    "details": kept
}
print("\nactivation 비교 결과를 GLOBAL_STORE['activation_cmp']에 저장했습니다.")


=== Activation Comparison (optimizer=RMSProp, dropout=0.5, batch=64) ===
activation  best_epoch  val_acc  val_macroF1  test_acc  test_macroF1
      relu           7   0.9381       0.9067    0.8943        0.8498
   sigmoid          49   0.7124       0.4444    0.7048        0.4321

Best by Val Macro-F1: relu

[VAL report — ReLU]
               precision    recall  f1-score   support

           0     0.8621    0.8333    0.8475        30
           1     0.9580    0.9856    0.9716       139
           2     0.9259    0.8772    0.9009        57

    accuracy                         0.9381       226
   macro avg     0.9153    0.8987    0.9067       226
weighted avg     0.9372    0.9381    0.9373       226

[TEST report — ReLU]
               precision    recall  f1-score   support

           0     0.8065    0.8065    0.8065        31
           1     0.9375    0.9712    0.9541       139
           2     0.8269    0.7544    0.7890        57

    accuracy                         0.8943      

In [10]:
# Cell 5

import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report

assert "GLOBAL_STORE" in globals(), "이전 셀을 먼저 실행해 GLOBAL_STORE를 준비하세요."

# 재현성
SEED = GLOBAL_STORE.get("SEED", 42)
np.random.seed(SEED); tf.random.set_seed(SEED)

# 데이터
X_train = GLOBAL_STORE["X_train"]; y_train = GLOBAL_STORE["y_train"]
X_val   = GLOBAL_STORE["X_val"];   y_val   = GLOBAL_STORE["y_val"]
X_test  = GLOBAL_STORE["X_test"];  y_test  = GLOBAL_STORE["y_test"]

input_dim = X_train.shape[1]
n_class   = 3

# 고정 하이퍼
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3
ACTIVATION = "relu"
DROPS      = [0.0, 0.3, 0.5, 0.7]

def build_mlp(input_dim, n_class, activation="relu", dropout=0.5):
    inputs = tf.keras.Input(shape=(input_dim,), name="tfidf")
    x = tf.keras.layers.Dense(512, activation=activation)(inputs)
    if dropout > 0: x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(256, activation=activation)(x)
    if dropout > 0: x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(n_class, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

def train_eval(drop):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = build_mlp(input_dim, n_class, activation=ACTIVATION, dropout=drop)
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    rlrop = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=0)

    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlrop],
        verbose=0
    )
    best_epoch = int(np.argmin(hist.history["val_loss"]) + 1)

    def eval_split(X, y):
        prob = model.predict(X, verbose=0)
        y_pred = prob.argmax(axis=1)
        return (
            accuracy_score(y, y_pred),
            f1_score(y, y_pred, average="macro"),
            classification_report(y, y_pred, digits=4, zero_division=0)
        )

    v_acc, v_f1, v_rep = eval_split(X_val, y_val)
    t_acc, t_f1, t_rep = eval_split(X_test, y_test)

    return {
        "dropout": drop,
        "best_epoch": best_epoch,
        "val_acc": round(v_acc,4), "val_macroF1": round(v_f1,4),
        "test_acc": round(t_acc,4), "test_macroF1": round(t_f1,4),
        "val_report": v_rep, "test_report": t_rep, "model": model
    }

results = []
kept = {}

for d in DROPS:
    out = train_eval(d)
    results.append({k: out[k] for k in ["dropout","best_epoch","val_acc","val_macroF1","test_acc","test_macroF1"]})
    kept[d] = out

df = pd.DataFrame(results).sort_values(by=["val_macroF1","val_acc"], ascending=False).reset_index(drop=True)
winner = df.loc[0, "dropout"]

print("=== Dropout Sweep (optimizer=RMSProp, activation=ReLU, batch=64) ===")
print(df.to_string(index=False))
print(f"\nBest by Val Macro-F1: dropout={winner}")

# 상세 리포트(베스트만 표시)
print("\n[VAL report — best]")
print(kept[winner]["val_report"])
print("\n[TEST report — best]")
print(kept[winner]["test_report"])

GLOBAL_STORE["dropout_cmp"] = {
    "results": df,
    "best": float(winner),
    "details": kept
}
print("\ndropout 비교 결과를 GLOBAL_STORE['dropout_cmp']에 저장했습니다.")


=== Dropout Sweep (optimizer=RMSProp, activation=ReLU, batch=64) ===
 dropout  best_epoch  val_acc  val_macroF1  test_acc  test_macroF1
     0.0           7   0.9513       0.9307    0.8987        0.8435
     0.5           7   0.9469       0.9192    0.8987        0.8517
     0.3           6   0.9425       0.9150    0.8987        0.8487
     0.7          13   0.9381       0.9040    0.8899        0.8398

Best by Val Macro-F1: dropout=0.0

[VAL report — best]
              precision    recall  f1-score   support

           0     0.9286    0.8667    0.8966        30
           1     0.9648    0.9856    0.9751       139
           2     0.9286    0.9123    0.9204        57

    accuracy                         0.9513       226
   macro avg     0.9406    0.9215    0.9307       226
weighted avg     0.9508    0.9513    0.9509       226


[TEST report — best]
              precision    recall  f1-score   support

           0     0.7500    0.7742    0.7619        31
           1     0.9448    0

In [11]:
# Cell 6


import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report

assert "GLOBAL_STORE" in globals(), "이전 셀들을 먼저 실행해 GLOBAL_STORE를 준비하세요."

# 재현성
SEED = GLOBAL_STORE.get("SEED", 42)
np.random.seed(SEED); tf.random.set_seed(SEED)

# 데이터
X_train = GLOBAL_STORE["X_train"]; y_train = GLOBAL_STORE["y_train"]
X_val   = GLOBAL_STORE["X_val"];   y_val   = GLOBAL_STORE["y_val"]
X_test  = GLOBAL_STORE["X_test"];  y_test  = GLOBAL_STORE["y_test"]

input_dim = X_train.shape[1]
n_class   = 3

# 고정 하이퍼
EPOCHS     = 50
LR         = 1e-3
ACTIVATION = "relu"
DROPOUT    = 0.0

BATCH_MINI = 64
BATCH_FULL = len(X_train)  # full-batch

def build_mlp(input_dim, n_class, activation="relu", dropout=0.0):
    inputs = tf.keras.Input(shape=(input_dim,), name="tfidf")
    x = tf.keras.layers.Dense(512, activation=activation)(inputs)
    # dropout=0.0 이면 생략
    x = tf.keras.layers.Dense(256, activation=activation)(x)
    outputs = tf.keras.layers.Dense(n_class, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

def train_eval(batch_size):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = build_mlp(input_dim, n_class, activation=ACTIVATION, dropout=DROPOUT)
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    rlrop = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=0)

    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=[es, rlrop],
        verbose=0
    )
    best_epoch = int(np.argmin(hist.history["val_loss"]) + 1)

    def eval_split(X, y):
        prob = model.predict(X, verbose=0)
        y_pred = prob.argmax(axis=1)
        return (
            accuracy_score(y, y_pred),
            f1_score(y, y_pred, average="macro"),
            classification_report(y, y_pred, digits=4, zero_division=0)
        )

    v_acc, v_f1, v_rep = eval_split(X_val, y_val)
    t_acc, t_f1, t_rep = eval_split(X_test, y_test)

    return {
        "batch_mode": ("mini-batch" if batch_size < len(X_train) else "full-batch"),
        "batch_size": int(batch_size),
        "best_epoch": best_epoch,
        "val_acc": round(v_acc,4), "val_macroF1": round(v_f1,4),
        "test_acc": round(t_acc,4), "test_macroF1": round(t_f1,4),
        "val_report": v_rep, "test_report": t_rep
    }

res_mini = train_eval(BATCH_MINI)
res_full = train_eval(BATCH_FULL)

df = pd.DataFrame([
    {k:res_mini[k] for k in ["batch_mode","batch_size","best_epoch","val_acc","val_macroF1","test_acc","test_macroF1"]},
    {k:res_full[k] for k in ["batch_mode","batch_size","best_epoch","val_acc","val_macroF1","test_acc","test_macroF1"]},
]).sort_values(by=["val_macroF1","val_acc"], ascending=False).reset_index(drop=True)

winner = df.loc[0, "batch_mode"]

print("=== Batch vs Mini-batch (optimizer=RMSProp, activation=ReLU, dropout=0.0) ===")
print(df.to_string(index=False))
print(f"\nBest by Val Macro-F1: {winner}")

# 상세 리포트(두 모드 모두 간단 확인)
print("\n[VAL report — mini-batch]\n", res_mini["val_report"])
print("[TEST report — mini-batch]\n", res_mini["test_report"])
print("\n[VAL report — full-batch]\n", res_full["val_report"])
print("[TEST report — full-batch]\n", res_full["test_report"])

GLOBAL_STORE["batch_cmp"] = {
    "results": df,
    "best": winner,
    "details": {"mini": res_mini, "full": res_full}
}
print("\n배치 크기 비교 결과를 GLOBAL_STORE['batch_cmp']에 저장했습니다.")


=== Batch vs Mini-batch (optimizer=RMSProp, activation=ReLU, dropout=0.0) ===
batch_mode  batch_size  best_epoch  val_acc  val_macroF1  test_acc  test_macroF1
mini-batch          64           7   0.9513       0.9307    0.9075        0.8702
full-batch        1811          50   0.9292       0.8966    0.8855        0.8105

Best by Val Macro-F1: mini-batch

[VAL report — mini-batch]
               precision    recall  f1-score   support

           0     0.9286    0.8667    0.8966        30
           1     0.9648    0.9856    0.9751       139
           2     0.9286    0.9123    0.9204        57

    accuracy                         0.9513       226
   macro avg     0.9406    0.9215    0.9307       226
weighted avg     0.9508    0.9513    0.9509       226

[TEST report — mini-batch]
               precision    recall  f1-score   support

           0     0.8621    0.8065    0.8333        31
           1     0.9257    0.9856    0.9547       139
           2     0.8800    0.7719    0.8224  

In [12]:
# Cell 7

import numpy as np, pandas as pd, tensorflow as tf, pickle, os
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

assert "GLOBAL_STORE" in globals(), "이전 셀들을 먼저 실행해 GLOBAL_STORE를 준비하세요."

# 재현성
SEED = GLOBAL_STORE.get("SEED", 42)
np.random.seed(SEED); tf.random.set_seed(SEED)

# 데이터 로딩
X_tr   = GLOBAL_STORE["X_train"]; y_tr   = GLOBAL_STORE["y_train"]
X_val  = GLOBAL_STORE["X_val"];   y_val  = GLOBAL_STORE["y_val"]
X_test = GLOBAL_STORE["X_test"];  y_test = GLOBAL_STORE["y_test"]
id2label = GLOBAL_STORE["id2label"]; label_names = [id2label[i] for i in sorted(id2label.keys())]
vectorizer = GLOBAL_STORE["vectorizer"]

# train+val 결합
X_train_full = np.concatenate([X_tr, X_val], axis=0)
y_train_full = np.concatenate([y_tr, y_val], axis=0)

input_dim = X_train_full.shape[1]
n_class   = len(label_names)

# 최종 베스트 설정
LR         = 1e-3
ACTIVATION = "relu"
DROPOUT    = 0.0
BATCH_SIZE = 64
EPOCHS     = 50

def build_mlp(input_dim, n_class, activation="relu", dropout=0.0):
    inputs = tf.keras.Input(shape=(input_dim,), name="tfidf")
    x = tf.keras.layers.Dense(512, activation=activation)(inputs)
    # dropout=0.0이면 생략
    x = tf.keras.layers.Dense(256, activation=activation)(x)
    outputs = tf.keras.layers.Dense(n_class, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
model = build_mlp(input_dim, n_class, activation=ACTIVATION, dropout=DROPOUT)
model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 콜백
es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlrop = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=0)
csv_logger = tf.keras.callbacks.CSVLogger("final_train_log.csv", append=False)

# 학습 (train+val에서 10%를 내부 검증으로 사용)
hist = model.fit(
    X_train_full, y_train_full,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es, rlrop, csv_logger],
    verbose=0,
    shuffle=True
)
best_epoch = int(np.argmin(hist.history["val_loss"]) + 1)
print(f"Final 재학습 완료 — best_epoch={best_epoch}, "
      f"val_loss@best={hist.history['val_loss'][best_epoch-1]:.4f}")

# 평가
proba = model.predict(X_test, verbose=0)
y_pred = proba.argmax(axis=1)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
report = classification_report(y_test, y_pred, target_names=label_names, digits=4, zero_division=0)
cm = confusion_matrix(y_test, y_pred, labels=list(range(n_class)))

print("\n=== FINAL Test 성능 (최종 설정: RMSProp, ReLU, Dropout=0.0, batch=64) ===")
print(f"TEST — Acc: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
print("\n[TEST Classification Report]\n" + report)
print("[TEST Confusion Matrix]\n", cm)

# 옵션: 아티팩트 저장
os.makedirs("artifacts", exist_ok=True)
model.save("artifacts/final_model.h5")
with open("artifacts/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# 요약 저장
summary = {
    "final_config": {
        "optimizer": "RMSProp", "lr": LR,
        "activation": ACTIVATION, "dropout": DROPOUT,
        "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "best_epoch": best_epoch
    },
    "test": {"accuracy": float(acc), "macro_f1": float(macro_f1)}
}
print("\nSaved: artifacts/final_model.h5, artifacts/vectorizer.pkl, final_train_log.csv")
print("최종 설정 요약:", summary)

GLOBAL_STORE["final"] = {"model": model, "summary": summary, "test_cm": cm, "test_report": report}
print("\nGLOBAL_STORE['final']에 결과를 저장했습니다.")


Final 재학습 완료 — best_epoch=10, val_loss@best=0.1646



=== FINAL Test 성능 (최종 설정: RMSProp, ReLU, Dropout=0.0, batch=64) ===
TEST — Acc: 0.9075 | Macro-F1: 0.8634

[TEST Classification Report]
              precision    recall  f1-score   support

    negative     0.8065    0.8065    0.8065        31
     neutral     0.9384    0.9856    0.9614       139
    positive     0.8800    0.7719    0.8224        57

    accuracy                         0.9075       227
   macro avg     0.8749    0.8547    0.8634       227
weighted avg     0.9057    0.9075    0.9053       227

[TEST Confusion Matrix]
 [[ 25   1   5]
 [  1 137   1]
 [  5   8  44]]

Saved: artifacts/final_model.h5, artifacts/vectorizer.pkl, final_train_log.csv
최종 설정 요약: {'final_config': {'optimizer': 'RMSProp', 'lr': 0.001, 'activation': 'relu', 'dropout': 0.0, 'batch_size': 64, 'epochs': 50, 'best_epoch': 10}, 'test': {'accuracy': 0.9074889867841409, 'macro_f1': 0.8634283427390707}}

GLOBAL_STORE['final']에 결과를 저장했습니다.


In [15]:
"""
데이터/세팅

Dataset: Financial PhraseBank - sentences_allagree (n=2,264)

Split: train/val/test = 1811/226/227 (stratified)

표현: TF-IDF (uni+bi, max_features=20k, min_df=2)

모델: MLP(512→256→softmax)

지표: Accuracy + Macro-F1 (불균형 대응)

1) Optimizer 비교 (ReLU, Dropout=0.5, batch=64)

RMSProp: Val Macro-F1=0.9274, Test Macro-F1=0.8507

Adam: 0.9224 / 0.8444

Momentum: 0.8892 / 0.8589

SGD: 0.2539 / 0.2532
결론: 수렴 안정성과 Val 기준에서 RMSProp 우세. Test에서 Momentum이 근소히 높았지만 전체 일관성/수렴 에폭을 고려해 RMSProp 채택.

2) Activation 비교 (optimizer=RMSProp, Dropout=0.5, batch=64)

ReLU: Val Macro-F1=0.9067, Test Macro-F1=0.8498

Sigmoid: 0.4444 / 0.4321
결론: ReLU 압승(기울기 소실 방지/빠른 수렴).

3) Dropout 스윕 (optimizer=RMSProp, activation=ReLU, batch=64)

0.0: Val Macro-F1=0.9307, Test Macro-F1=0.8435

0.3: 0.9150 / 0.8487

0.5: 0.9192 / 0.8517

0.7: 0.9040 / 0.8398
결론: Val 기준 Dropout=0.0 우세(데이터 규모/모델 용량 대비 정규화 과다 시 성능 하락). Test에선 0.5가 약간 높았지만, 전체 실험 계획상 Val 우선으로 0.0 채택.

4) Batch vs Mini-batch (RMSProp, ReLU, Dropout=0.0)

Mini-batch(64): Val Macro-F1=0.9307, Test=0.8702

Full-batch: 0.8966 / 0.8105
결론: Mini-batch가 일반화·안정성 모두 우위.

5) 최종 모델 (train+val 재학습 → test)

설정: RMSProp(lr=1e-3) + ReLU + Dropout=0.0 + batch=64, best_epoch=10

FINAL Test — Acc: 0.9075 | Macro-F1: 0.8634

혼동행렬:

negative: 25 / 31 맞춤

neutral: 137 / 139 맞춤

positive: 44 / 57 맞춤
결론(한 줄): RMSProp + ReLU + 미니배치(64) + Dropout 미적용이 이 데이터에서 가장 안정적이고 높은 Macro-F1을 달성했다 (0.8634)
"""


'\n데이터/세팅\n\nDataset: Financial PhraseBank - sentences_allagree (n=2,264)\n\nSplit: train/val/test = 1811/226/227 (stratified)\n\n표현: TF-IDF (uni+bi, max_features=20k, min_df=2)\n\n모델: MLP(512→256→softmax)\n\n지표: Accuracy + Macro-F1 (불균형 대응)\n\n1) Optimizer 비교 (ReLU, Dropout=0.5, batch=64)\n\nRMSProp: Val Macro-F1=0.9274, Test Macro-F1=0.8507\n\nAdam: 0.9224 / 0.8444\n\nMomentum: 0.8892 / 0.8589\n\nSGD: 0.2539 / 0.2532\n결론: 수렴 안정성과 Val 기준에서 RMSProp 우세. Test에서 Momentum이 근소히 높았지만 전체 일관성/수렴 에폭을 고려해 RMSProp 채택.\n\n2) Activation 비교 (optimizer=RMSProp, Dropout=0.5, batch=64)\n\nReLU: Val Macro-F1=0.9067, Test Macro-F1=0.8498\n\nSigmoid: 0.4444 / 0.4321\n결론: ReLU 압승(기울기 소실 방지/빠른 수렴).\n\n3) Dropout 스윕 (optimizer=RMSProp, activation=ReLU, batch=64)\n\n0.0: Val Macro-F1=0.9307, Test Macro-F1=0.8435\n\n0.3: 0.9150 / 0.8487\n\n0.5: 0.9192 / 0.8517\n\n0.7: 0.9040 / 0.8398\n결론: Val 기준 Dropout=0.0 우세(데이터 규모/모델 용량 대비 정규화 과다 시 성능 하락). Test에선 0.5가 약간 높았지만, 전체 실험 계획상 Val 우선으로 0.0 채택.\n\n4) Batch vs Mini-